# LangChain Deep Technical Blog – HuggingFace Version

This notebook uses local HuggingFace models via Transformers + LangChain wrappers to avoid OpenAI billing/quota issues.

In [ ]:
# Install Dependencies
!pip install -U langchain langchain-community langchain-huggingface transformers accelerate faiss-cpu sentence-transformers


## Import Libraries

In [2]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate


In [3]:
import os
from getpass import getpass

os.environ["HF_TOKEN"] = getpass("Enter Hugging Face Token: ")

Enter Hugging Face Token: ··········


## Load Local HuggingFace Model

In [ ]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline

hf_pipeline = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    token=os.environ["HF_TOKEN"],
    max_new_tokens=100,
    temperature=0.3
)

model = HuggingFacePipeline(pipeline=hf_pipeline)

## Basic LLM Call

In [5]:
response = model.invoke(
    "Question: Explain Prompt Engineering in simple terms.\nAnswer:"
)

print(response)

Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: Explain Prompt Engineering in simple terms.
Answer: Prompt Engineering is a process of creating a product or service that meets the needs of the market in a timely and cost-effective manner. It involves identifying the needs of the market, developing a prototype, testing the prototype, refining the product or service, and launching it into the market. The process is iterative and continuous, with continuous feedback and improvement.


## Prompt Template

In [6]:
prompt = PromptTemplate.from_template(
    "Explain {topic} in simple terms."
)

formatted = prompt.invoke({"topic": "Vector Databases"})
print(formatted)


text='Explain Vector Databases in simple terms.'


## Simple Chain

In [7]:
chain = prompt | model

response = chain.invoke({"topic": "Prompt Engineering"})
print(response)


Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Explain Prompt Engineering in simple terms.


## Memory Simulation

In [8]:
chat_history = []

chat_history.append(("User", "Hi"))
chat_history.append(("AI", "Hello!"))
chat_history.append(("User", "What is LangChain?"))

print(chat_history)


[('User', 'Hi'), ('AI', 'Hello!'), ('User', 'What is LangChain?')]


## Custom Tool

In [9]:
def calculator_tool(expression: str):
    return eval(expression)

print(calculator_tool("25 * 8"))


200


## Embeddings

In [ ]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
embedding = embed_model.encode("What is LangChain?")
print(len(embedding))


## FAISS Vector Store

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

texts = [
    "LangChain is a framework for LLM applications.",
    "FAISS is used for vector similarity search."
]

vectorstore = FAISS.from_texts(texts, embedding_model)

results = vectorstore.similarity_search("What is LangChain?")
print(results[0].page_content)


## Mini RAG Example

In [12]:
context = results[0].page_content

rag_prompt = PromptTemplate.from_template(
    "Answer the question based on context.\nContext: {context}\nQuestion: {question}"
)

rag_chain = rag_prompt | model

answer = rag_chain.invoke({
    "context": context,
    "question": "Explain LangChain"
})

print(answer)


Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer the question based on context.
Context: LangChain is a framework for LLM applications.
Question: Explain LangChain's approach to language modeling.


## Use Cases

In [13]:
use_cases = [
    "Customer Support Chatbot",
    "PDF Question Answering",
    "Research Assistant"
]

for case in use_cases:
    print("-", case)


- Customer Support Chatbot
- PDF Question Answering
- Research Assistant


## Conclusion
This notebook demonstrates LangChain concepts using HuggingFace local/free models, avoiding OpenAI quota issues.